In [ ]:
# =============================================================================
# CELL 1: Imports & Configuration
# =============================================================================
import os, glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import brier_score_loss, mutual_info_score
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# --- Data paths ---
DATA_DIR = r"Data\RFpredictions"
MIN_ROWS = 100
MIN_TICKERS_PER_DATE = 20

# --- Risk parameters (from Util.py STRATEGY_PARAMS + 9_SuperFastBroker.py) ---
RISK_PARAMS = {
    'stop_loss_pct': 5.0,           # Fixed stop loss %
    'take_profit_pct': 20.0,        # Fixed take profit %
    'trailing_stop_base': 1.5,      # Trailing stop base %
    'trailing_stop_slope': 0.75,    # Trailing stop ATR scaling
    'trailing_stop_cap': 5.0,       # Max trailing stop %
    'position_timeout': 5,          # Max hold days
    'expected_profit_per_day': 0.25 # Min daily return %
}

# --- Benchmark ---
RISK_FREE_RATE = 0.035  # 3.5% annualized

# --- Analysis horizon ---
HORIZONS = [1, 3, 5]
PRIMARY_HORIZON = 1  # 1-day ahead focus

print("Configuration")
print("=" * 60)
print(f"Data directory:    {DATA_DIR}")
print(f"Primary horizon:   {PRIMARY_HORIZON}-day ahead")
print(f"Risk-free rate:    {RISK_FREE_RATE:.1%}")
print(f"Stop loss:         {RISK_PARAMS['stop_loss_pct']}%")
print(f"Take profit:       {RISK_PARAMS['take_profit_pct']}%")
print(f"Trailing stop:     {RISK_PARAMS['trailing_stop_base']}% + {RISK_PARAMS['trailing_stop_slope']}*max(0, ATR%-2%), cap {RISK_PARAMS['trailing_stop_cap']}%")
print(f"Position timeout:  {RISK_PARAMS['position_timeout']} days")

In [ ]:
# =============================================================================
# CELL 2: Load All Prediction Files
# =============================================================================

def load_all_predictions(folder):
    frames, failed = [], []
    files = glob.glob(os.path.join(folder, "*.parquet"))
    print(f"Found {len(files)} parquet files")

    for i, path in enumerate(files, 1):
        if i % 500 == 0:
            print(f"  Loading {i}/{len(files)}...")
        try:
            d = pd.read_parquet(path)
            required = {"Date", "Close", "UpProbability"}
            if not required.issubset(d.columns):
                failed.append((path, "Missing columns"))
                continue
            if len(d) < MIN_ROWS:
                failed.append((path, f"Only {len(d)} rows"))
                continue
            if not pd.api.types.is_datetime64_any_dtype(d['Date']):
                d['Date'] = pd.to_datetime(d['Date'])
            d = d.sort_values("Date")
            d["Ticker"] = os.path.basename(path).replace(".parquet", "")
            frames.append(d)
        except Exception as e:
            failed.append((path, str(e)))

    print(f"Loaded {len(frames)} files | Failed: {len(failed)}")
    if failed and len(failed) <= 10:
        for p, r in failed:
            print(f"  - {os.path.basename(p)}: {r}")

    combined = pd.concat(frames, ignore_index=True).sort_values(['Ticker', 'Date'])
    print(f"\nDataset: {len(combined):,} rows | {combined['Ticker'].nunique()} tickers")
    print(f"Date range: {combined['Date'].min().date()} to {combined['Date'].max().date()}")
    return combined

df = load_all_predictions(DATA_DIR)

In [ ]:
# =============================================================================
# CELL 3: Compute Forward Returns & Risk-Adjusted Trade Returns
# =============================================================================

def compute_atr(group, period=14):
    """Compute ATR per ticker group"""
    high = group['High'].astype(float)
    low = group['Low'].astype(float)
    close = group['Close'].astype(float)
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def simulate_trade_return(row, df_ticker, idx, params):
    """
    Simulate a single trade using broker risk rules.
    Entry at Close on signal day, then track over hold period.
    Returns the capped trade return and exit reason.
    """
    entry_price = float(row['Close'])
    if entry_price <= 0:
        return np.nan, 'Bad Price'

    atr = row.get('ATR', entry_price * 0.02)
    if pd.isna(atr) or atr <= 0:
        atr = entry_price * 0.02

    # Stop loss
    stop_price = entry_price * (1 - params['stop_loss_pct'] / 100)

    # Take profit (from broker: risk * 2.0 RR or fixed %)
    risk = entry_price - stop_price
    tp_rr = entry_price + risk * 2.0
    tp_fixed = entry_price * (1 + params['take_profit_pct'] / 100)
    take_profit = min(tp_rr, tp_fixed)

    # Dynamic trailing stop % (from 9_SuperFastBroker._calculate_dynamic_trail)
    atr_pct = (atr / entry_price) * 100
    trail_pct = params['trailing_stop_base'] + params['trailing_stop_slope'] * max(0, atr_pct - 2.0)
    trail_pct = min(trail_pct, params['trailing_stop_cap'])

    # Simulate forward days
    max_days = params['position_timeout']
    high_water = entry_price
    trailing_stop = entry_price * (1 - trail_pct / 100)

    for d in range(1, max_days + 1):
        future_idx = idx + d
        if future_idx >= len(df_ticker):
            # Early exit - use last available close
            exit_price = float(df_ticker.iloc[min(future_idx, len(df_ticker)-1)]['Close'])
            ret = (exit_price / entry_price) - 1
            return ret, 'Data End'

        future_row = df_ticker.iloc[future_idx]
        day_high = float(future_row['High'])
        day_low = float(future_row['Low'])
        day_close = float(future_row['Close'])

        # Check take profit (intraday high)
        if day_high >= take_profit:
            ret = (take_profit / entry_price) - 1
            return ret, 'Take Profit'

        # Check stop loss (intraday low)
        if day_low <= stop_price:
            ret = (stop_price / entry_price) - 1
            return ret, 'Stop Loss'

        # Update trailing stop
        if day_high > high_water:
            high_water = day_high
            trailing_stop = high_water * (1 - trail_pct / 100)

        # Check trailing stop
        if day_low <= trailing_stop:
            ret = (trailing_stop / entry_price) - 1
            return ret, 'Trailing Stop'

        # Check poor performance (after day 2)
        profit_pct = ((day_close / entry_price) - 1) * 100
        min_expected = d * params['expected_profit_per_day']
        if d > 2 and profit_pct < min_expected:
            ret = (day_close / entry_price) - 1
            return ret, 'Poor Performance'

    # Timeout exit
    exit_price = float(df_ticker.iloc[min(idx + max_days, len(df_ticker)-1)]['Close'])
    ret = (exit_price / entry_price) - 1
    return ret, 'Timeout'


# Compute ATR per ticker
df['ATR'] = df.groupby('Ticker').apply(
    lambda g: compute_atr(g)
).reset_index(level=0, drop=True)

# Compute simple forward returns for IC analysis
for h in HORIZONS:
    df[f'fwd_close_{h}'] = df.groupby('Ticker')['Close'].shift(-h)
    df[f'fwd_return_{h}'] = (df[f'fwd_close_{h}'] / df['Close'].astype(float)) - 1
    df[f'fwd_up_{h}'] = (df[f'fwd_return_{h}'] > 0).astype(int)

# Simulate risk-adjusted trade returns for signals above threshold
print("Simulating risk-adjusted trades...")
trade_returns = []
trade_reasons = []

threshold = df['PositiveThreshold'].dropna().mode()
if len(threshold) > 0:
    threshold = float(threshold.iloc[0])
else:
    threshold = 0.60

print(f"Using signal threshold: {threshold}")

for ticker, g in df.groupby('Ticker'):
    g = g.reset_index(drop=True)
    for idx, row in g.iterrows():
        if row['UpProbability'] >= threshold:
            ret, reason = simulate_trade_return(row, g, idx, RISK_PARAMS)
            trade_returns.append(ret)
            trade_reasons.append(reason)
        else:
            trade_returns.append(np.nan)
            trade_reasons.append(None)

df['trade_return'] = trade_returns
df['exit_reason'] = trade_reasons

# Clean
df_clean = df.dropna(subset=['UpProbability', f'fwd_return_{PRIMARY_HORIZON}'])

# Stats
n_trades = df['trade_return'].notna().sum()
print(f"\nForward returns computed for horizons: {HORIZONS}")
print(f"Risk-simulated trades: {n_trades:,} (signals >= {threshold})")
print(f"Dataset after cleaning: {len(df_clean):,} rows")

In [ ]:
# =============================================================================
# CELL 4: Information Coefficient & Mutual Information
# =============================================================================

def compute_ic_table(df, horizons):
    results = []
    for h in horizons:
        ret_col = f'fwd_return_{h}'
        valid = df.dropna(subset=[ret_col, 'UpProbability'])
        ic_s, p_s = spearmanr(valid['UpProbability'], valid[ret_col])
        ic_p, p_p = pearsonr(valid['UpProbability'], valid[ret_col])
        n = len(valid)
        t_s = ic_s * np.sqrt((n - 2) / (1 - ic_s**2))
        results.append({
            'Horizon': f'{h}d',
            'Spearman IC': ic_s,
            't-stat': t_s,
            'Pearson IC': ic_p,
            'N': n
        })
    return pd.DataFrame(results)

def compute_mutual_info(df, horizon):
    ret_col = f'fwd_return_{horizon}'
    valid = df.dropna(subset=[ret_col, 'UpProbability'])

    # Discretized MI (binned returns)
    n_bins = 10
    ret_binned = pd.qcut(valid[ret_col], q=n_bins, labels=False, duplicates='drop')
    prob_binned = pd.qcut(valid['UpProbability'], q=n_bins, labels=False, duplicates='drop')
    mi_discrete = mutual_info_score(prob_binned, ret_binned)

    # Continuous MI via sklearn
    mi_continuous = mutual_info_regression(
        valid[['UpProbability']].values,
        valid[ret_col].values,
        n_neighbors=5,
        random_state=42
    )[0]

    return mi_discrete, mi_continuous

def compute_daily_rank_ic(df, horizon):
    ret_col = f'fwd_return_{horizon}'
    daily_ics, dates = [], []
    for date, group in df.groupby('Date'):
        if len(group) < MIN_TICKERS_PER_DATE:
            continue
        ic, _ = spearmanr(group['UpProbability'], group[ret_col])
        if not np.isnan(ic):
            daily_ics.append(ic)
            dates.append(date)
    return pd.DataFrame({'Date': dates, 'Rank_IC': daily_ics})


# IC table
ic_table = compute_ic_table(df_clean, HORIZONS)
print("Information Coefficient (UpProbability vs Forward Returns)")
print("=" * 70)
print(ic_table.to_string(index=False))

# Mutual Information
mi_disc, mi_cont = compute_mutual_info(df_clean, PRIMARY_HORIZON)
print(f"\nMutual Information ({PRIMARY_HORIZON}-day)")
print("-" * 40)
print(f"  Discrete MI (binned):    {mi_disc:.4f}")
print(f"  Continuous MI (KNN):     {mi_cont:.4f}")
print(f"  Interpretation:          {'Signal present' if mi_cont > 0.001 else 'Weak/no signal'}")

# Daily IC
daily_ic = compute_daily_rank_ic(df_clean, PRIMARY_HORIZON)
pct_positive = (daily_ic['Rank_IC'] > 0).mean()
print(f"\nDaily Cross-Sectional Rank IC ({PRIMARY_HORIZON}-day)")
print("-" * 40)
print(f"  Mean IC:          {daily_ic['Rank_IC'].mean():.4f}")
print(f"  Std IC:           {daily_ic['Rank_IC'].std():.4f}")
print(f"  % Positive Days:  {pct_positive:.1%}")
print(f"  ICIR:             {daily_ic['Rank_IC'].mean() / daily_ic['Rank_IC'].std():.3f}")

# Plots
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# IC decay
ax = axes[0, 0]
ax.bar([f'{h}d' for h in HORIZONS], ic_table['Spearman IC'], color='steelblue', alpha=0.8, edgecolor='black')
for i, v in enumerate(ic_table['Spearman IC']):
    ax.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_title('Spearman IC by Horizon', fontweight='bold')
ax.set_ylabel('IC')
ax.grid(axis='y', alpha=0.3)

# Daily IC time series
ax = axes[0, 1]
roll = daily_ic.set_index('Date')['Rank_IC'].rolling(20).mean()
ax.fill_between(daily_ic['Date'], daily_ic['Rank_IC'], alpha=0.15, color='blue')
ax.plot(roll.index, roll.values, color='darkblue', lw=2, label='20d Rolling Mean')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_title(f'Daily Rank IC ({PRIMARY_HORIZON}d)', fontweight='bold')
ax.set_ylabel('IC')
ax.legend()
ax.grid(alpha=0.3)

# IC distribution
ax = axes[1, 0]
ax.hist(daily_ic['Rank_IC'], bins=40, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(daily_ic['Rank_IC'].mean(), color='red', ls='--', lw=2, label=f"Mean={daily_ic['Rank_IC'].mean():.4f}")
ax.axvline(0, color='black', ls='-', lw=1)
ax.set_title('Daily IC Distribution', fontweight='bold')
ax.set_xlabel('Rank IC')
ax.legend()
ax.grid(alpha=0.3)

# MI bar
ax = axes[1, 1]
ax.bar(['Discrete MI', 'Continuous MI'], [mi_disc, mi_cont], color=['coral', 'seagreen'], alpha=0.8, edgecolor='black')
for i, v in enumerate([mi_disc, mi_cont]):
    ax.text(i, v + 0.0002, f'{v:.4f}', ha='center', fontsize=10)
ax.set_title(f'Mutual Information ({PRIMARY_HORIZON}d)', fontweight='bold')
ax.set_ylabel('MI (nats)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 5: Risk-Adjusted Returns (Sharpe, Omega, Sortino) with Trade Simulation
# =============================================================================

def compute_sharpe(returns, rf_annual=RISK_FREE_RATE, periods_per_year=252):
    """Annualized Sharpe ratio"""
    if len(returns) < 2 or returns.std() == 0:
        return 0.0
    rf_per_period = rf_annual / periods_per_year
    excess = returns - rf_per_period
    return (excess.mean() / excess.std()) * np.sqrt(periods_per_year)

def compute_sortino(returns, rf_annual=RISK_FREE_RATE, periods_per_year=252):
    """Annualized Sortino ratio (downside deviation)"""
    if len(returns) < 2:
        return 0.0
    rf_per_period = rf_annual / periods_per_year
    excess = returns - rf_per_period
    downside = excess[excess < 0]
    if len(downside) == 0 or downside.std() == 0:
        return np.inf if excess.mean() > 0 else 0.0
    return (excess.mean() / downside.std()) * np.sqrt(periods_per_year)

def compute_omega(returns, threshold=0.0):
    """
    Omega ratio: sum of gains above threshold / sum of losses below threshold.
    Omega > 1 means the strategy is profitable vs the threshold.
    """
    gains = returns[returns > threshold] - threshold
    losses = threshold - returns[returns <= threshold]
    if losses.sum() == 0:
        return np.inf if gains.sum() > 0 else 1.0
    return gains.sum() / losses.sum()

def compute_max_drawdown(returns):
    """Max drawdown from cumulative return series"""
    cum = (1 + returns).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min()

# --- Analyze risk-adjusted trades ---
trades = df[df['trade_return'].notna()].copy()
trade_rets = trades['trade_return']

# Daily risk-free threshold for Omega
rf_daily = RISK_FREE_RATE / 252

# Compute metrics
sharpe = compute_sharpe(trade_rets)
sortino = compute_sortino(trade_rets)
omega = compute_omega(trade_rets, threshold=rf_daily)
omega_zero = compute_omega(trade_rets, threshold=0.0)
max_dd = compute_max_drawdown(trade_rets)
win_rate = (trade_rets > 0).mean()
avg_win = trade_rets[trade_rets > 0].mean() if (trade_rets > 0).any() else 0
avg_loss = trade_rets[trade_rets <= 0].mean() if (trade_rets <= 0).any() else 0
profit_factor = abs(avg_win * (trade_rets > 0).sum()) / abs(avg_loss * (trade_rets <= 0).sum()) if avg_loss != 0 else np.inf

print("RISK-ADJUSTED TRADE PERFORMANCE")
print("=" * 60)
print(f"  Total Trades:         {len(trades):,}")
print(f"  Mean Return:          {trade_rets.mean():.4%}")
print(f"  Median Return:        {trade_rets.median():.4%}")
print(f"  Std Dev:              {trade_rets.std():.4%}")
print(f"  Win Rate:             {win_rate:.2%}")
print(f"  Avg Win:              {avg_win:.4%}")
print(f"  Avg Loss:             {avg_loss:.4%}")
print(f"  Profit Factor:        {profit_factor:.2f}")
print(f"  Max Drawdown:         {max_dd:.4%}")
print(f"\n  Sharpe Ratio (ann):   {sharpe:.3f}")
print(f"  Sortino Ratio (ann):  {sortino:.3f}")
print(f"  Omega (vs 0%):        {omega_zero:.3f}")
print(f"  Omega (vs Rf/day):    {omega:.3f}")

# Exit reason breakdown
print(f"\nExit Reason Breakdown:")
print("-" * 40)
reason_stats = trades.groupby('exit_reason')['trade_return'].agg(['count', 'mean', 'median'])
reason_stats.columns = ['Count', 'Mean Return', 'Median Return']
reason_stats = reason_stats.sort_values('Count', ascending=False)
print(reason_stats.to_string())

# --- Visualizations ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Trade return distribution
ax = axes[0, 0]
ax.hist(trade_rets.clip(-0.15, 0.25), bins=60, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', ls='--', lw=2)
ax.axvline(trade_rets.mean(), color='green', ls='--', lw=2, label=f'Mean={trade_rets.mean():.3%}')
ax.set_title('Trade Return Distribution', fontweight='bold')
ax.set_xlabel('Return')
ax.legend()
ax.grid(alpha=0.3)

# 2. Cumulative equity curve
ax = axes[0, 1]
cum = (1 + trade_rets.reset_index(drop=True)).cumprod()
ax.plot(cum, color='darkblue', lw=1.5)
ax.axhline(1, color='red', ls='--', lw=1)
ax.set_title('Cumulative Equity (Trade Sequence)', fontweight='bold')
ax.set_ylabel('Growth of $1')
ax.set_xlabel('Trade #')
ax.grid(alpha=0.3)

# 3. Exit reason pie
ax = axes[0, 2]
reason_counts = trades['exit_reason'].value_counts()
colors_map = {'Take Profit': 'green', 'Stop Loss': 'red', 'Trailing Stop': 'orange',
              'Timeout': 'gray', 'Poor Performance': 'salmon', 'Data End': 'lightblue'}
pie_colors = [colors_map.get(r, 'purple') for r in reason_counts.index]
ax.pie(reason_counts, labels=reason_counts.index, autopct='%1.1f%%', colors=pie_colors, textprops={'fontsize': 8})
ax.set_title('Exit Reasons', fontweight='bold')

# 4. Mean return by exit reason
ax = axes[1, 0]
reason_means = trades.groupby('exit_reason')['trade_return'].mean().sort_values()
colors_bar = ['green' if v > 0 else 'red' for v in reason_means]
ax.barh(reason_means.index, reason_means.values, color=colors_bar, alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', ls='-', lw=1)
ax.set_title('Mean Return by Exit Reason', fontweight='bold')
ax.set_xlabel('Mean Return')
ax.grid(axis='x', alpha=0.3)

# 5. Rolling Sharpe (50-trade window)
ax = axes[1, 1]
window = min(50, len(trade_rets) // 5)
if window > 5:
    rolling_sharpe = trade_rets.rolling(window).apply(lambda x: compute_sharpe(x), raw=False)
    ax.plot(rolling_sharpe.reset_index(drop=True), color='darkblue', lw=1.5)
    ax.axhline(0, color='red', ls='--', lw=1)
    ax.set_title(f'Rolling Sharpe ({window}-trade window)', fontweight='bold')
    ax.set_ylabel('Sharpe')
    ax.set_xlabel('Trade #')
    ax.grid(alpha=0.3)

# 6. Win rate by probability decile
ax = axes[1, 2]
trades['prob_decile'] = pd.qcut(trades['UpProbability'], q=10, labels=False, duplicates='drop')
decile_wr = trades.groupby('prob_decile')['trade_return'].apply(lambda x: (x > 0).mean())
ax.bar(decile_wr.index, decile_wr.values, color='seagreen', alpha=0.7, edgecolor='black')
ax.axhline(0.5, color='red', ls='--', lw=1)
ax.set_title('Win Rate by Probability Decile', fontweight='bold')
ax.set_xlabel('Decile (0=low prob, 9=high)')
ax.set_ylabel('Win Rate')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 6: Regime-Conditional Analysis (VIX Regimes)
# =============================================================================

def regime_analysis(df, trades_df):
    if 'VIX_Close' not in df.columns:
        print("VIX_Close not available - skipping regime analysis")
        return None

    valid = trades_df.dropna(subset=['trade_return']).copy()
    if 'VIX_Close' not in valid.columns:
        valid = valid.merge(df[['Date', 'Ticker', 'VIX_Close']].drop_duplicates(),
                            on=['Date', 'Ticker'], how='left', suffixes=('', '_merge'))
        if 'VIX_Close_merge' in valid.columns:
            valid['VIX_Close'] = valid['VIX_Close'].fillna(valid['VIX_Close_merge'])

    valid = valid.dropna(subset=['VIX_Close'])
    if len(valid) < 100:
        print("Not enough data with VIX for regime analysis")
        return None

    terciles = valid['VIX_Close'].quantile([0.33, 0.67])

    def classify(v):
        if v <= terciles.iloc[0]: return 'Low VIX'
        elif v <= terciles.iloc[1]: return 'Med VIX'
        else: return 'High VIX'

    valid['regime'] = valid['VIX_Close'].apply(classify)

    results = []
    for regime in ['Low VIX', 'Med VIX', 'High VIX']:
        rd = valid[valid['regime'] == regime]
        if len(rd) < 20:
            continue
        rets = rd['trade_return']
        results.append({
            'Regime': regime,
            'Trades': len(rd),
            'Mean Return': rets.mean(),
            'Win Rate': (rets > 0).mean(),
            'Sharpe': compute_sharpe(rets),
            'Omega': compute_omega(rets, threshold=0),
            'Sortino': compute_sortino(rets),
            'VIX Range': f"{rd['VIX_Close'].min():.1f}-{rd['VIX_Close'].max():.1f}"
        })

    regime_df = pd.DataFrame(results)
    print("Regime-Conditional Performance (Risk-Adjusted Trades)")
    print("=" * 80)
    print(regime_df.to_string(index=False))
    return regime_df, valid

result = regime_analysis(df, trades)
if result:
    regime_df, regime_trades = result

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Sharpe by regime
    ax = axes[0]
    colors = ['green', 'gold', 'red']
    ax.bar(regime_df['Regime'], regime_df['Sharpe'], color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(0, color='black', ls='--', lw=1)
    ax.set_title('Sharpe Ratio by VIX Regime', fontweight='bold')
    ax.set_ylabel('Annualized Sharpe')
    ax.grid(axis='y', alpha=0.3)

    # Omega by regime
    ax = axes[1]
    ax.bar(regime_df['Regime'], regime_df['Omega'], color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(1, color='red', ls='--', lw=1, label='Break-even')
    ax.set_title('Omega Ratio by VIX Regime', fontweight='bold')
    ax.set_ylabel('Omega')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # Return distribution by regime
    ax = axes[2]
    for regime, color in zip(['Low VIX', 'Med VIX', 'High VIX'], colors):
        rd = regime_trades[regime_trades['regime'] == regime]['trade_return']
        ax.hist(rd.clip(-0.1, 0.15), bins=40, alpha=0.4, label=regime, color=color, density=True)
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.set_title('Return Distribution by Regime', fontweight='bold')
    ax.set_xlabel('Trade Return')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# CELL 7: Parameter Sensitivity Sweep (Stop Loss / Take Profit / Trailing)
# =============================================================================

def sweep_params(df, base_params, param_name, values, signal_threshold):
    """Sweep a single parameter and compute metrics for each value"""
    results = []
    for val in values:
        params = base_params.copy()
        params[param_name] = val

        all_rets = []
        for ticker, g in df.groupby('Ticker'):
            g = g.reset_index(drop=True)
            for idx, row in g.iterrows():
                if row['UpProbability'] >= signal_threshold:
                    ret, _ = simulate_trade_return(row, g, idx, params)
                    if not np.isnan(ret):
                        all_rets.append(ret)

        rets = pd.Series(all_rets)
        if len(rets) < 10:
            continue

        results.append({
            param_name: val,
            'Trades': len(rets),
            'Mean Return': rets.mean(),
            'Win Rate': (rets > 0).mean(),
            'Sharpe': compute_sharpe(rets),
            'Omega': compute_omega(rets, threshold=0),
            'Sortino': compute_sortino(rets),
            'Max DD': compute_max_drawdown(rets),
        })

    return pd.DataFrame(results)

print("Running parameter sensitivity sweeps...")
print("(This may take a few minutes with ~4000 files)\n")

# Use a sample for speed if dataset is large
if df['Ticker'].nunique() > 200:
    sample_tickers = df['Ticker'].drop_duplicates().sample(200, random_state=42)
    df_sample = df[df['Ticker'].isin(sample_tickers)].copy()
    print(f"Using {df_sample['Ticker'].nunique()} ticker sample for sweep")
else:
    df_sample = df.copy()

# Sweep stop loss
print("Sweeping stop_loss_pct...")
sl_sweep = sweep_params(df_sample, RISK_PARAMS, 'stop_loss_pct',
                        [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0], threshold)

# Sweep take profit
print("Sweeping take_profit_pct...")
tp_sweep = sweep_params(df_sample, RISK_PARAMS, 'take_profit_pct',
                        [5.0, 8.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0], threshold)

# Sweep trailing stop base
print("Sweeping trailing_stop_base...")
trail_sweep = sweep_params(df_sample, RISK_PARAMS, 'trailing_stop_base',
                           [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0], threshold)

# Sweep position timeout
print("Sweeping position_timeout...")
timeout_sweep = sweep_params(df_sample, RISK_PARAMS, 'position_timeout',
                             [1, 2, 3, 4, 5, 7, 10], threshold)

# Print results
for name, sweep in [('Stop Loss %', sl_sweep), ('Take Profit %', tp_sweep),
                     ('Trailing Stop Base %', trail_sweep), ('Position Timeout (days)', timeout_sweep)]:
    print(f"\n{name} Sensitivity:")
    print("=" * 80)
    print(sweep.to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

sweeps = [
    ('stop_loss_pct', sl_sweep, 'Stop Loss %'),
    ('take_profit_pct', tp_sweep, 'Take Profit %'),
    ('trailing_stop_base', trail_sweep, 'Trail Stop Base %'),
    ('position_timeout', timeout_sweep, 'Hold Period (days)')
]

for col, (param, sweep, label) in enumerate(sweeps):
    if sweep.empty:
        continue
    x = sweep[param]

    # Sharpe
    ax = axes[0, col]
    ax.plot(x, sweep['Sharpe'], 'o-', color='darkblue', lw=2, markersize=6)
    best_idx = sweep['Sharpe'].idxmax()
    ax.axvline(sweep.loc[best_idx, param], color='green', ls='--', alpha=0.5,
               label=f'Best={sweep.loc[best_idx, param]}')
    ax.set_title(f'Sharpe vs {label}', fontweight='bold')
    ax.set_ylabel('Sharpe')
    ax.set_xlabel(label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Omega
    ax = axes[1, col]
    ax.plot(x, sweep['Omega'], 's-', color='coral', lw=2, markersize=6)
    ax.axhline(1, color='red', ls='--', lw=1)
    best_idx = sweep['Omega'].idxmax()
    ax.axvline(sweep.loc[best_idx, param], color='green', ls='--', alpha=0.5,
               label=f'Best={sweep.loc[best_idx, param]}')
    ax.set_title(f'Omega vs {label}', fontweight='bold')
    ax.set_ylabel('Omega')
    ax.set_xlabel(label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Parameter Sensitivity Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Optimal parameters summary
print("\n" + "=" * 60)
print("OPTIMAL PARAMETERS (by Sharpe)")
print("=" * 60)
for param, sweep, label in sweeps:
    if sweep.empty:
        continue
    best = sweep.loc[sweep['Sharpe'].idxmax()]
    print(f"  {label:25s}: {best[param]:>6}  (Sharpe={best['Sharpe']:.3f}, Omega={best['Omega']:.3f}, WR={best['Win Rate']:.1%})")

In [ ]:
# =============================================================================
# CELL 8: Comprehensive Summary Report
# =============================================================================

print("=" * 80)
print(" PREDICTION + RISK MANAGEMENT EVALUATION REPORT")
print("=" * 80)

# 1. IC
print("\n1. INFORMATION COEFFICIENT")
print("-" * 60)
for _, row in ic_table.iterrows():
    quality = "Very Strong" if row['Spearman IC'] > 0.06 else \
              "Strong" if row['Spearman IC'] > 0.03 else \
              "Tradeable" if row['Spearman IC'] > 0.01 else "Weak"
    print(f"  {row['Horizon']:>3s}:  IC={row['Spearman IC']:.4f}  t={row['t-stat']:.1f}  [{quality}]")

# 2. Mutual Information
print("\n2. MUTUAL INFORMATION")
print("-" * 60)
print(f"  Discrete MI:    {mi_disc:.4f}")
print(f"  Continuous MI:  {mi_cont:.4f}")

# 3. Temporal Stability
print("\n3. TEMPORAL STABILITY")
print("-" * 60)
print(f"  Mean Daily IC:     {daily_ic['Rank_IC'].mean():.4f}")
print(f"  ICIR:              {daily_ic['Rank_IC'].mean() / daily_ic['Rank_IC'].std():.3f}")
print(f"  % Positive Days:   {(daily_ic['Rank_IC'] > 0).mean():.1%}")

# 4. Risk-Adjusted Performance
print("\n4. RISK-ADJUSTED TRADE PERFORMANCE")
print("-" * 60)
print(f"  Trades:            {len(trades):,}")
print(f"  Mean Return:       {trade_rets.mean():.4%}")
print(f"  Win Rate:          {win_rate:.2%}")
print(f"  Profit Factor:     {profit_factor:.2f}")
print(f"  Sharpe (ann):      {sharpe:.3f}")
print(f"  Sortino (ann):     {sortino:.3f}")
print(f"  Omega (vs 0):      {omega_zero:.3f}")
print(f"  Max Drawdown:      {max_dd:.4%}")

# 5. Risk Param Assessment
print("\n5. RISK PARAMETER ASSESSMENT")
print("-" * 60)
reason_pcts = trades['exit_reason'].value_counts(normalize=True)
for reason, pct in reason_pcts.items():
    avg_r = trades[trades['exit_reason'] == reason]['trade_return'].mean()
    print(f"  {reason:20s}: {pct:>6.1%} of exits  (avg return: {avg_r:+.3%})")

# 6. Regime
if result:
    print("\n6. REGIME PERFORMANCE")
    print("-" * 60)
    for _, row in regime_df.iterrows():
        print(f"  {row['Regime']:10s}: Sharpe={row['Sharpe']:>6.3f}  Omega={row['Omega']:>5.3f}  WR={row['Win Rate']:.1%}")

# 7. Scorecard
print("\n7. OVERALL SCORECARD")
print("-" * 60)

checks = [
    ("IC > 0.02 (1d)",           ic_table[ic_table['Horizon'] == f'{PRIMARY_HORIZON}d']['Spearman IC'].iloc[0] > 0.02),
    ("IC t-stat > 3",            ic_table[ic_table['Horizon'] == f'{PRIMARY_HORIZON}d']['t-stat'].iloc[0] > 3),
    (">55% positive IC days",    pct_positive > 0.55),
    ("MI > 0.001",               mi_cont > 0.001),
    ("Trade Win Rate > 52%",     win_rate > 0.52),
    ("Sharpe > 0.5",            sharpe > 0.5),
    ("Omega > 1.0",             omega_zero > 1.0),
    ("Max DD > -20%",           max_dd > -0.20),
]

passed = sum(1 for _, v in checks if v)
for name, v in checks:
    status = "PASS" if v else "FAIL"
    print(f"  {name:30s}  {status}")

print(f"\n  Score: {passed}/{len(checks)}")

if passed >= 7:
    grade = "EXCELLENT - Deploy with confidence"
elif passed >= 6:
    grade = "STRONG - Ready for live trading"
elif passed >= 5:
    grade = "GOOD - Minor refinements needed"
elif passed >= 4:
    grade = "FAIR - Review risk params"
else:
    grade = "WEAK - Major revision needed"

print(f"  Grade: {grade}")
print("\n" + "=" * 80)